# Expected Loss as an Integral: From Calculus to Risk-Aware Machine Learning in FinTech

## 0. Вступление

Во многих системах принятия решений задача выглядит довольно просто:
нужно определить, стоит ли предпринимать действие для конкретного клиента.

Во многих системах принятия решений задача выглядит довольно просто:
нужно определить, стоит ли предпринимать действие для конкретного клиента.

Например, в B2B-дистрибуции компания может предложить скидку, персональные условия или дополнительное внимание со стороны менеджера, чтобы предотвратить отток клиента. Однако любое действие имеет цену. Скидка снижает маржу, дополнительная работа менеджера требует времени, а специальные условия могут влиять на экономику всей клиентской базы.

Поэтому решение всегда сводится к сравнению двух величин:

- ожидаемого убытка, если клиент уйдёт

- стоимости действия по удержанию клиента

В самой простой модели это можно записать следующим образом:

$$EV = P(churn) \cdot LTV - Cost(action)$$

где:

- $P(churn) -$ вероятность оттока клиента
- $LTV -$ ожидаемая потеря выручки
- $Cost(action) - $ стоимость удерживающего действия

Однако такая формула предполагает, что потери фиксированы.
На практике это почти никогда не так.

Даже если клиент действительно уйдёт, величина ущерба может сильно различаться. Иногда клиент сокращает оборот лишь немного, иногда полностью прекращает закупки, а иногда перестаёт работать с компанией постепенно. Величина потерь $L$ является случайной величиной, а не фиксированным числом.

Если потери имеют распределение с плотностью $f(L∣churn)$, то ожидаемый ущерб определяется уже не одной константой, а математическим ожиданием случайной величины:
$$E[L] = \int_{0}^{\infty} L f(L | churn) dL$$

Это означает, что для оценки ожидаемого ущерба нужно интегрировать возможные значения потерь по их распределению.

С учётом вероятности оттока ожидаемый убыток клиента можно записать так:

$$E[Loss] = p \cdot \int_{0}^{\infty} L f (L | churn) dL$$

где $p = P(churn)$

Тогда правило принятия решения принимает следующий вид:

$$Act \ if \ p \cdot \int_{0}^{\infty} Lf(L | churn) dL > C$$

где $C -$ стоимость действия по удержанию клиента.

Таким образом, решение системы сводится к сравнению двух величин:

- ожидаемого убытка, вычисленного через интеграл по распределению потерь

- стоимости действия

Подобная конструкция лежит в основе многих задач decision making under uncertainty и risk-aware систем принятия решений.

Интересно, что за этим довольно прикладным правилом стоит один из фундаментальных объектов математического анализа — определённый (а в данном случае несобственный) интеграл.

В этой статье мы последовательно разберём:

- как интеграл появляется в модели ожидаемого убытка

- как он связан с математическим ожиданием и плотностями распределения

- как его можно вычислять аналитически, численно и через симуляции

- как такие идеи используются в машинном обучении и системах принятия решений

В итоге мы увидим, что даже простая бизнес-формула для оценки риска опирается на довольно фундаментальные математические идеи, лежащие в основе современных ML-систем.

## План статьи 

**1. Problem Framing**

Начнём с простой бизнес-задачи:
как определить, стоит ли предпринимать действие для удержания клиента.

Покажем, что решение зависит от сравнения:

ожидаемого убытка от оттока

стоимости действия

И сформулируем decision rule.

**2. Потери как случайная величина**

Обсудим, почему потери клиента редко являются фиксированной величиной.

Покажем, что ущерб лучше моделировать как случайную величину с распределением.

Это приводит к формуле математического ожидания через интеграл.

**3. Интеграл как ожидание**

Кратко введём определённый интеграл и покажем, как он используется для вычисления математического ожидания непрерывной случайной величины.

Покажем, что в задачах риска часто возникает несобственный интеграл.

**4. Аналитический пример**

На простом распределении потерь покажем, как математическое ожидание можно вычислить аналитически.

Это позволит увидеть структуру формулы expected loss.

**5. Численное интегрирование**

Покажем, как тот же интеграл можно вычислить численно с помощью Python.

Это приближает математическую модель к реальной инженерной практике.

**6. Monte Carlo симуляция**

Рассмотрим альтернативный способ оценки ожидания через симуляцию случайных выборок.

Покажем связь между интегралом и средним по выборке.

**7. Использование ML-инструментов**

Покажем, как современные ML-библиотеки позволяют работать с такими моделями и оптимизировать параметры с помощью автоматического дифференцирования.

**8. Небольшое математическое упражнение**

Разберём задачу уровня вступительных экзаменов Школы анализа данных:
как проверить, является ли функция плотностью распределения.

Это позволит глубже понять роль интеграла в вероятностных моделях.

**9. Связь с машинным обучением**

Покажем, что теоретический риск модели также выражается через интеграл, а используемый на практике эмпирический риск является его дискретной аппроксимацией.

**10. Возвращение к системе принятия решений**

В конце вернёмся к исходной задаче и покажем, как интеграл ожидания превращается в правило принятия решения для системы вроде Stabrium.

## 1. Переход к Problem Framing

До этого момента мы обсуждали проблему на интуитивном уровне:
если ожидаемый ущерб от оттока клиента превышает стоимость удерживающего действия, то действие имеет смысл.

Однако чтобы превратить эту интуицию в работающую систему принятия решений, нужно формализовать несколько вещей:

- как именно измеряется риск потери клиента

- как оценить величину возможного ущерба

- как объединить эти величины в единое правило принятия решения

Другими словами, нам нужно аккуратно сформулировать decision problem.

Такое формальное описание особенно важно в системах, где решения принимаются автоматически — например, в риск-моделях, системах предотвращения оттока или recommendation-движках. 

В этих системах решение должно основываться не на одной величине, а на сочетании вероятности события и распределения его последствий.

Именно здесь естественным образом появляется интеграл.

1. Decision Problem

Рассмотрим типичную задачу в системах управления клиентским риском.

Для каждого клиента система должна определить, стоит ли предпринимать действие по удержанию — например, предложить скидку или специальное условие.

Для этого нам необходимо оценить три величины:

1. Вероятность события

Вероятность того, что клиент уйдёт:

$$p = P(churn | x)$$

где $x -$ признаки клиента

Эта вероятность обычно оценивается моделью машинного обучения.

2. Потенциальный ущерб

Если клиент действительно уйдёт, компания понесёт некоторый убыток $L$
Однако величина этого убытка заранее неизвестна и может сильно варьироваться. Поэтому $L$ удобно рассматривать как случайную величину, имеющую распределение с плотностью

$$f(L∣churn)$$

Такое распределение описывает возможные сценарии потерь:
от небольшого снижения оборота до полной потери клиента.

3. Стоимость действия

Любое действие по удержанию клиента имеет цену:

$$C$$

Это может быть стоимость скидки, дополнительного сервиса или времени менеджера.

### **Expected Loss**

Если клиент уходит с вероятностью $p$, а величина потерь имеет распределение 
$f(L∣churn)$, то ожидаемый убыток клиента равен

$$E[Loss]=p \cdot E[L]$$

где

$$E[L] = \int_{0}^{\infty} L f(L | churn) dL $$

 математическое ожидание потерь.

Этот интеграл агрегирует все возможные значения ущерба с учётом их вероятностей.

### **Decision Rule**

Теперь правило принятия решения можно записать в строгой форме:

$$\text{Act \ if \ } \ p \cdot \int_{0}^{\infty} L f (L | churn) dL > C$$

То есть система предпринимает действие тогда, когда ожидаемый ущерб превышает стоимость вмешательства.

Это правило лежит в основе многих risk-aware decision systems.

Переход к математике

Чтобы применять это правило на практике, необходимо научиться вычислять величину

$$E[L] = \int_{0}^{\infty} L f(L | churn) dL $$


Здесь появляется ключевой объект нашей статьи — определённый (а точнее, несобственный) интеграл.

## 2. Потери как случайная величина

В предыдущем разделе мы сформулировали правило принятия решения через ожидаемый убыток клиента. Однако в этой формуле скрыто важное предположение: величина потерь должна быть корректно определена.

На первый взгляд можно было бы считать, что потеря клиента равна некоторой фиксированной величине — например, его текущему обороту или ожидаемому LTV. Такая модель иногда используется на практике из-за своей простоты. Тем не менее она плохо отражает реальную динамику бизнеса.

Даже если клиент действительно прекращает сотрудничество, последствия могут быть очень разными. Один клиент может лишь временно снизить объём закупок, другой полностью прекращает работу с компанией, а третий может вернуться спустя некоторое время. Кроме того, величина потерь зависит от множества факторов: структуры заказов, маржинальности продукции, сезонности и длительности отношений с клиентом.

Поэтому в более реалистичной модели потери 
𝐿
L рассматриваются не как фиксированное число, а как случайная величина. Иначе говоря, при оттоке клиента возможны различные значения ущерба, каждое из которых имеет свою вероятность.

Если распределение этих потерь описывается плотностью
$$f(L∣churn)$$

то функция $f(L∣churn)$ характеризует вероятность различных сценариев ущерба при оттоке клиента. Такая модель позволяет учитывать неопределённость и естественным образом агрегировать разные возможные исходы.

В этом случае одной величины $L$ уже недостаточно. Нас интересует средний ожидаемый ущерб, который возникает при оттоке клиента.

Для непрерывной случайной величины математическое ожидание определяется через интеграл:

$$E[L] = \int_{0}^{\infty} Lf(L | churn) dL$$

Интуитивно этот интеграл суммирует все возможные значения потерь, взвешивая их по вероятности появления.

Важно заметить, что пределы интегрирования здесь не ограничены сверху. Это означает, что мы имеем дело с несобственным интегралом, который определяется как предел

$$E[L] = \lim_{b \to \infty} \int_{0}^{\infty} Lf(L | churn) dL$$

Такая запись гарантирует корректность определения математического ожидания даже в случае, когда возможные значения потерь теоретически не ограничены.

Таким образом, переход от фиксированной величины потерь к случайной величине приводит нас к центральному объекту этой статьи — интегралу, который агрегирует неопределённость распределения ущерба.

В следующем разделе мы подробнее рассмотрим, как именно интеграл используется для вычисления математического ожидания и как его можно вычислять на практике.

## **3. Интеграл как ожидание**

Структурно здесь происходит следующее:

определение интеграла → из матанализа (Пискунов)

интерпретация как математического ожидания → из теории вероятностей

несобственный интеграл → снова матанализ

###  1. Постановка задачи. Нижняя и верхняя интегральные суммы

<div align="center">
  <img src="image.png" alt="alt text" width="800">
</div>

# Построение нижних и верхних интегральных сумм

Пусть на отрезке $[a,b]$ задана непрерывная функция  
$y = f(x)$ (см. рис. 210–211).

Обозначим через $m$ и $M$ соответственно её наименьшее и наибольшее значения на этом отрезке.

---

## Разбиение отрезка

Разобьём отрезок $[a,b]$ на $n$ частей точками

$$
a = x_0, x_1, x_2, \dots, x_{n-1}, x_n = b,
$$

где

$$
x_0 < x_1 < x_2 < \dots < x_n.
$$

Обозначим длины частичных отрезков:

$$
\Delta x_1 = x_1 - x_0,
$$

$$
\Delta x_2 = x_2 - x_1,
$$

$$
\dots
$$

$$
\Delta x_n = x_n - x_{n-1}.
$$

---

## Локальные минимумы и максимумы

Обозначим:

- через $m_1$ и $M_1$ — наименьшее и наибольшее значения функции $f(x)$ на $[x_0, x_1]$;
- через $m_2$ и $M_2$ — наименьшее и наибольшее значения на $[x_1, x_2]$;
- $\dots$
- через $m_n$ и $M_n$ — наименьшее и наибольшее значения на $[x_{n-1}, x_n]$.

---

## Нижняя и верхняя интегральные суммы

Составим суммы.

### Нижняя интегральная сумма:

$$
s_n =
m_1 \Delta x_1 +
m_2 \Delta x_2 +
\dots +
m_n \Delta x_n
=
\sum_{i=1}^{n} m_i \Delta x_i.
$$

### Верхняя интегральная сумма:

$$
S_n =
M_1 \Delta x_1 +
M_2 \Delta x_2 +
\dots +
M_n \Delta x_n
=
\sum_{i=1}^{n} M_i \Delta x_i.
$$

# Нижние и верхние интегральные суммы

Сумму $s_n$ называют **нижней интегральной суммой**,  
а сумму $\overline{s}_n$ — **верхней интегральной суммой**.

Если $f(x) \ge 0$, то нижняя интегральная сумма численно равняется площади
вписанной ступенчатой фигуры  
$AC_0N_1C_1N_2 \dots C_{n-1}N_nBA$,  
ограниченной «вписанной» ломаной.

Верхняя интегральная сумма численно равняется площади «описанной» ступенчатой фигуры  
$AK_0C_1K_1 \dots C_{n-1}K_{n-1}C_nBA$,  
ограниченной «описанной» ломаной.

---

## Свойства верхних и нижних интегральных сумм

### а)

Так как $m_i \le M_i$ для любого $i = 1,2,\dots,n$,  
то на основании формул (1) и (2) имеем

$$
s_n \le \overline{s}_n.
\tag{3}
$$

(Знак равенства возможен только при $f(x)=\text{const}$.)

---

### б)

Пусть $m$ — наименьшее значение функции $f(x)$ на $[a,b]$.

Так как
$$
m_1 \ge m,\quad m_2 \ge m,\quad \dots,\quad m_n \ge m,
$$

то

$$
s_n = m_1\Delta x_1 + m_2\Delta x_2 + \dots + m_n\Delta x_n
$$

$$
\ge
m\Delta x_1 + m\Delta x_2 + \dots + m\Delta x_n
=
m(\Delta x_1 + \dots + \Delta x_n)
=
m(b-a).
$$

Следовательно,

$$
s_n \ge m(b-a).
\tag{4}
$$

---

### в)

Пусть $M$ — наибольшее значение функции $f(x)$ на $[a,b]$.

Так как
$$
M_1 \le M,\quad M_2 \le M,\quad \dots,\quad M_n \le M,
$$

то

$$
\overline{s}_n =
M_1\Delta x_1 + M_2\Delta x_2 + \dots + M_n\Delta x_n
$$

$$
\le
M\Delta x_1 + M\Delta x_2 + \dots + M\Delta x_n
=
M(\Delta x_1 + \dots + \Delta x_n)
=
M(b-a).
$$

Следовательно,

$$
\overline{s}_n \le M(b-a).
\tag{5}
$$

---

## Итоговое неравенство

Объединяя полученные результаты, получаем

$$
m(b-a) \le s_n \le \overline{s}_n \le M(b-a).
\tag{6}
$$

Если $f(x) \ge 0$, то это неравенство имеет простой геометрический смысл:
величины $m(b-a)$ и $M(b-a)$ равны соответственно площадям
«вписанного» и «описанного» прямоугольников.


<div align="center">
  <img src="image-1.png" alt="alt text" width="400">
</div>